# image_hidream

**HiDream-O1-Image-Dev** 로 World Reference → 장소 배경 → 인물 초상 → 합성 확인.

| | |
|---|---|
| 커널 | `img` |
| 출력 | `outputs/image_hidream/<시나리오>/{world_ref, places, portraits, composites}/` |

## FLUX.2 klein은 제외했다

klein의 텍스트 인코더가 `Qwen3-4B-FP8` 인데, **FP8 연산은 compute capability 8.9 이상
(4090/H100급)에서만 된다.** RTX 3090은 8.6이라 하드웨어적으로 못 돌린다.
4090 이상을 쓰게 되면 그때 비교군으로 다시 붙이면 된다.

## Dev 변형을 쓰는 이유

full은 24GB에 빠듯하다. Dev는 약 10GB / 28스텝이라 3090에 여유롭다.
**Dev는 CFG가 꺼져 있어 네거티브 프롬프트가 무효**다. 원하지 않는 요소는
전부 긍정 프롬프트로 눌러야 한다.


## 1. 커널 확인

설치는 `setup_kernels.ipynb` 가 한다. 커널을 `img` 로 바꾸고 실행할 것.

In [ ]:
# ---------------------------------------------------------------------------
# 커널 확인. 설치는 setup_kernels.ipynb 가 담당한다.
#
# 이 셀에 pip install 을 추가하지 말 것. 노트북 4개는 서로 양립 불가능한
# torch/transformers 를 쓰기 때문에, 여기서 다시 깔면 커널 조합이 깨진다.
#   ace  torch 2.10.0+cu128 / transformers >=4.51,<4.58
#   sa3  torch 2.7.1 cu126  / transformers >=5.8
#   flux torch 2.8.0        / transformers ==4.56.1
# ---------------------------------------------------------------------------
import sys, importlib

KERNEL = "img"
NEED   = ["torch", "diffusers", "transformers", "PIL"]

print("python     :", sys.version.split()[0])
print("interpreter:", sys.executable)
print()

missing = []
for m in NEED:
    try:
        mod = importlib.import_module(m)
        print(f"OK   {m:16s} {getattr(mod, '__version__', '')}")
    except Exception as e:
        missing.append(m)
        print(f"FAIL {m:16s} {type(e).__name__}: {e}")

assert not missing, (
    f"\n{missing} 를 import 하지 못했다.\n"
    f"커널을 '{KERNEL}' 로 바꿀 것 (Kernel > Change kernel).\n"
    f"'{KERNEL}' 커널이 목록에 없으면 setup_kernels.ipynb 를 먼저 실행한다.\n"
    f"여기서 pip install 로 때우지 말 것 — 버전 조합이 깨진다."
)
print("\n커널 OK")


# HiDreamO1ImagePipeline 은 diffusers 에 없는 이름이다.
# 문서상 존재하는 클래스는 HiDreamImagePipeline 하나이고,
# 지원 체크포인트는 HiDream-I1-Full / Dev / Fast 다.
from diffusers import HiDreamImagePipeline
print("HiDreamImagePipeline 사용 가능")

## 2. 설정


In [ ]:
import os, sys, json, pathlib, getpass, shutil

USER = getpass.getuser()

# ---- 이 노드에서 유효한 루트를 직접 찾는다 --------------------------------
# 학과 클러스터는 같은 저장소가 노드마다 다른 경로에 마운트된다.
#   로그인 노드(hpcmaster) : /data1/$USER   (= HOME)
#   계산 노드(hpc-stat1)   : /mnt/data1/$USER
# setup_kernels 로 만든 커널이면 PROJ_BASE 가 kernel.json 에 박혀 있고,
# Slurm 으로 돌리면 common.sh 가 환경변수로 넘겨준다. 둘 다 아니면 여기서 찾는다.
def pick_root():
    for c in (f"/mnt/data1/{USER}", f"/data1/{USER}",
              str(pathlib.Path.home()), "/workspace", "/scratch"):
        if pathlib.Path(c).is_dir() and os.access(c, os.W_OK):
            return c
    return None

ROOT = os.environ.get("PROJ_ROOT") or pick_root()
if ROOT is None:
    raise SystemExit(f"쓸 수 있는 루트를 못 찾았다 (user={USER})")

# 계산 노드에는 $HOME 이 아예 없다. 두면 캐시/kernelspec 이 전부 깨진다.
if not pathlib.Path(os.environ.get("HOME", "/nonexistent")).is_dir():
    os.environ["HOME"] = ROOT

BASE  = pathlib.Path(os.environ.get("PROJ_BASE") or f"{ROOT}/mystery")
MODEL = "image_hidream"
OUT_DIR = BASE / "outputs" / MODEL

# 캐시는 홈이 아니라 대용량 스토리지로. 홈 쿼터면 다운로드 중 os error 28 이 난다.
os.environ.setdefault("HF_HOME",      str(BASE / "hf_cache"))
os.environ.setdefault("TMPDIR",       str(BASE / "tmp"))
os.environ.setdefault("MPLCONFIGDIR", str(BASE / "mpl_cache"))

try:
    for p in (OUT_DIR, pathlib.Path(os.environ["HF_HOME"]),
              pathlib.Path(os.environ["TMPDIR"])):
        p.mkdir(parents=True, exist_ok=True)
except (PermissionError, OSError) as e:
    raise SystemExit(
        f"{BASE} 에 쓸 수 없다 -> {type(e).__name__}: {e}\n"
        f"  이 셀 맨 위에서 지정할 것:\n"
        f"      os.environ['PROJ_BASE'] = '/data1/{USER}/mystery'"
    )

# ---- 입력 JSON (scenarios_en/*.json 전부) ---------------------------------
# papermill 로 돌리면 cwd 가 /tmp 라 "." 만으로는 못 찾는다.
# PROJ_INPUT(환경변수) 을 먼저 보고, 없으면 후보들을 뒤진다.
def pick_input_dir():
    cands = []
    if os.environ.get("PROJ_INPUT"):
        cands.append(pathlib.Path(os.environ["PROJ_INPUT"]))
    cands += [pathlib.Path.cwd(), pathlib.Path.cwd().parent, BASE]
    for c in cands:
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    # 마지막 수단: ROOT 아래를 얕게 훑는다
    for c in pathlib.Path(ROOT).glob("*/*/"):
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    return None

IN_DIR = pick_input_dir()
if IN_DIR is None:
    raise SystemExit(
        "scenarios_en/*.json 을 못 찾았다.\n"
        f"  PROJ_INPUT = {os.environ.get('PROJ_INPUT', '(미설정)')}\n"
        f"  cwd        = {pathlib.Path.cwd()}\n"
        f"  이 셀 맨 위에서:  os.environ['PROJ_INPUT'] = '/data1/{USER}/<저장소>/Jaemin'"
    )

# scenario_id(파일 내부 필드, 예: "12_heungbu")를 키로 쓴다 — 파일명과 무관하게 안정적.
SCENARIO_FILES = sorted((IN_DIR / "scenarios_en").glob("*.json"))
assert SCENARIO_FILES, f"scenarios_en 에 json 이 없다: {IN_DIR / 'scenarios_en'}"

DATA = {}
for _f in SCENARIO_FILES:
    _d = json.load(open(_f, encoding="utf-8"))
    DATA[_d["scenario_id"]] = _d

for k in DATA:
    (OUT_DIR / k).mkdir(parents=True, exist_ok=True)


def out_path(scen, *parts):
    """OUT_DIR/<시나리오>/<...> 경로를 만들고 돌려준다."""
    p = OUT_DIR / scen
    for x in parts[:-1]:
        p = p / x
    p.mkdir(parents=True, exist_ok=True)
    return p / parts[-1]


print("node    :", os.uname().nodename)
print("ROOT    :", ROOT)
print("BASE    :", BASE, f"(여유 {shutil.disk_usage(BASE).free/1e9:.0f} GB)")
print("IN_DIR  :", IN_DIR)
print("HF_HOME :", os.environ["HF_HOME"])
print("OUT_DIR :", OUT_DIR)
print(f"시나리오 {len(DATA)}개:")
for k, d in DATA.items():
    print(f"  {k:24s} 장소 {len(d['places'])} / 인물 {len(d['cast'])}")


In [ ]:
import gc, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print("torch  :", torch.__version__)
print("device :", DEVICE)
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("GPU    :", p.name, "|", round(p.total_memory / 1e9, 1), "GB",
          "| compute capability", f"{p.major}.{p.minor}")
    if (p.major, p.minor) < (8, 9):
        print("  주의: FP8 연산 미지원 세대. FP8 가중치를 요구하는 모델은 못 돌린다.")
else:
    print("  GPU가 없다. 로그인 노드이거나 GPU 할당이 안 된 상태.")


def free():
    """모델을 내린 뒤 호출. 커널을 종료하는 것이 가장 확실하다."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        print(f"할당된 VRAM: {used:.2f} GB")


## 3. 프롬프트 조립


In [ ]:
# 두 이미지 노트북(image_hidream / image_flux2_klein)에 동일하게 들어간다.
# 프롬프트가 달라지면 모델 비교가 성립하지 않는다.
#
# STYLE: HiDream 쪽 파이프라인은 레퍼런스 이미지를 받지 못한다
#        (HiDreamImagePipeline.__call__ 에 image 인자가 없다).
#        그래서 세계관 앵커를 문장으로 고정한다. FLUX 는 이 문장 + world_ref
#        이미지를 함께 쓴다.
STYLE = ("cohesive painterly realism, muted desaturated palette, "
         "soft directional daylight, film grain, no text, no watermark")


def build_prompts(d):
    era      = d["setting"]["era"]
    loc      = d["setting"]["location"]
    tone     = d["adapted_story"]["tone"]
    synopsis = d["adapted_story"]["synopsis"]
    palette  = d["adapted_story"]["palette"]

    world = (f"empty, uninhabited establishing reference image — no people, "
             f"no humans, no figures, no bodies, no corpse, nobody present "
             f"anywhere in frame. defines the visual world. "
             f"period and place: {era}, {loc}. mood: {tone}. "
             f"color palette: {palette}. consistent color palette, lighting "
             f"logic and material language. completely vacant, deserted, "
             f"empty of any person or body no matter what else is implied. "
             f"{STYLE}.")

    places = {}
    for p in d["places"]:
        clue_list = "; ".join(p["clues"])
        places[p["place_id"]] = (
            f"empty, uninhabited space — no people, no humans, no figures, "
            f"no bodies, no corpse, nobody present anywhere in frame. "
            f"{p['name']}. {p['description']} "
            f"this scene must clearly show: {clue_list} — as objects and "
            f"traces only, with no person or body present to go with them. "
            f"period and place: {era}, {loc}. mood: {tone}. "
            f"color palette: {palette}. "
            f"completely vacant, deserted, unoccupied, empty of any person "
            f"or body no matter what else is implied. {STYLE}.")

    # 이름은 절대 넣지 않는다. 외모 묘사만으로 그린다.
    portraits = {c["id"]: (f"portrait of a {c['age']}-year-old {c['gender']}, "
                           f"{c['occupation']}. {c['appearance']} "
                           f"context: {synopsis} "
                           f"setting: {era}. mood: {tone}. "
                           f"color palette: {palette}. "
                           f"waist-up framing, neutral background, even lighting. "
                           f"{STYLE}.")
                 for c in d["cast"]}
    return {"world": world, "places": places, "portraits": portraits}


P = {k: build_prompts(d) for k, d in DATA.items()}

# 시나리오마다 자기 source_title 단어를 금지어로 쓴다 — 하드코딩된 2개짜리
# 리스트 대신, 20개 전부에서 원작 고유명사가 새어 들어갔는지 자동으로 잡는다.
STUDIO_TERMS = ["disney", "pixar", "dreamworks", "ghibli"]

STOPWORDS = {
    "the", "a", "an", "of", "in", "on", "for", "with", "to", "and", "or",
    "korean", "folktale", "fairy", "tale", "tales", "myth", "legend", "legends",
    "novel", "classical", "pansori", "based", "original", "setting", "custom",
    "little", "big", "great", "old", "young", "new",
    "girl", "boy", "man", "woman", "king", "queen", "prince", "princess",
    "red", "white", "black", "gold", "silver", "snow",
    "hood", "riding", "match", "sun", "moon", "woodcutter",
    "night", "day", "house", "story",
    # 일반 명사인데 우연히 제목에 들어간 것들 (실제 IP 고유명사가 아님)
    "piper", "office", "startup", "theater", "theatre", "silent-film", "silent", "film",
}

def banned_terms(d):
    import re
    title = re.sub(r"\(.*?\)", "", d.get("source_title", ""))
    words = {w.strip(".,").lower() for w in title.split() if len(w) > 2}
    return (words - STOPWORDS) | set(STUDIO_TERMS)

import re

bad = []
for scen, v in P.items():
    blob = " ".join([v["world"]] + list(v["places"].values())
                    + list(v["portraits"].values())).lower()
    # 단어 경계로 검사한다 — "pied" 가 "unoccupied" 안의 부분문자열로 잡히는 식의
    # 오탐(substring match)을 막는다.
    hits = [b for b in banned_terms(DATA[scen])
            if re.search(rf"\b{re.escape(b)}\b", blob)]
    if hits:
        bad.append((scen, hits))

for scen, v in P.items():
    print("=" * 70); print(scen)
    print("  [world]", v["world"][:120], "...")
    print(f"  장소 {len(v['places'])} / 인물 {len(v['portraits'])}")

print()
print("이름 유입 검사:", "OK" if not bad else f"FAIL {bad}")
assert not bad, "ENRICH 의 occupation / location / place name 을 확인할 것"
total_places = sum(len(v["places"]) for v in P.values())
total_portraits = sum(len(v["portraits"]) for v in P.values())
print(f"생성 예정: world {len(P)}장 + 장소 {total_places}장 + 인물 {total_portraits}장 "
      f"= {len(P) + total_places + total_portraits}장")

## 4. 모델 로드

처음 실행하면 가중치를 내려받는다. 디스크 여유를 먼저 확인할 것.


In [ ]:
!df -h $HF_HOME | tail -1


In [ ]:
# ---------------------------------------------------------------------------
# HiDream-I1-Dev 로드
#
# 주의 1. HiDreamImagePipeline 은 text_encoder_4 로 Llama-3.1-8B-Instruct 를
#         따로 받아야 하는데, HiDream-I1-Dev 저장소에는 text_encoder_4 가 없다
#         (text_encoder / _2 / _3 만 들어 있다).
# 주의 2. meta-llama/Llama-3.1-8B-Instruct 는 gated(수동 승인)다.
#         승인 전이면 아래 LLAMA_ID 를 미러로 바꾼다.
# 주의 3. 가중치 합계가 60GB 를 넘는다. 24GB GPU 에서는 CPU 오프로딩이 필수다.
# ---------------------------------------------------------------------------
import torch
from transformers import AutoTokenizer, LlamaForCausalLM
from diffusers import HiDreamImagePipeline

MODEL_ID = "HiDream-ai/HiDream-I1-Dev"
LLAMA_ID = os.environ.get("LLAMA_ID", "meta-llama/Llama-3.1-8B-Instruct")
# 승인 대기 중이면:
# LLAMA_ID = "NousResearch/Meta-Llama-3.1-8B-Instruct"

STEPS, CFG, SIZE = 28, 0.0, 1024   # Dev 는 distilled. CFG=0 이라 네거티브는 무효다.

print("텍스트 인코더 4 (Llama):", LLAMA_ID)
tokenizer_4 = AutoTokenizer.from_pretrained(LLAMA_ID)
text_encoder_4 = LlamaForCausalLM.from_pretrained(
    LLAMA_ID,
    output_hidden_states=True,
    output_attentions=True,
    torch_dtype=DTYPE,
)

pipe = HiDreamImagePipeline.from_pretrained(
    MODEL_ID,
    tokenizer_4=tokenizer_4,
    text_encoder_4=text_encoder_4,
    torch_dtype=DTYPE,
)
pipe.enable_model_cpu_offload()     # 24GB 에서는 필수. 느려지지만 OOM 을 막는다.


def gen(prompt, seed=0, size=SIZE):
    """레퍼런스 이미지는 받지 않는다.
       HiDreamImagePipeline.__call__ 에 image 인자가 없다. 예전 코드처럼
       image=... 를 넘기면 **kwargs 로 조용히 버려져서 (TypeError 도 안 난다)
       전혀 다른 그림이 나온다. 그래서 아예 인자를 두지 않는다.
       세계관 일관성은 프롬프트의 STYLE 문장과 고정 seed 로 맞춘다."""
    g = torch.Generator("cpu").manual_seed(seed)   # cpu offload 라 generator 도 cpu
    return pipe(
        prompt=prompt,
        height=size, width=size,
        num_inference_steps=STEPS,
        guidance_scale=CFG,
        generator=g,
    ).images[0]


print("준비됨:", MODEL_ID)

## 5. World Reference

**반드시 먼저.** 이걸 레퍼런스로 걸어야 장소끼리 같은 세계로 보인다.


In [ ]:
# world_ref 는 레퍼런스로 걸 수 없다 (파이프라인이 image 를 안 받는다).
# 그래도 "이 세계가 어떻게 생겼는가" 를 눈으로 확인하는 기준으로는 필요하다.
WR = {}
for scen, v in P.items():
    img = gen(v["world"], seed=0)
    img.save(out_path(scen, "world_ref.png"))
    WR[scen] = img
    print("world_ref:", scen)

## 6. 장소 배경


In [ ]:
for scen, v in P.items():
    for pid, prompt in v["places"].items():
        gen(prompt, seed=0).save(out_path(scen, "places", f"{pid}.png"))
        print("  ", scen, pid)

## 7. 인물 초상 + 표정

베이스 1장을 고정하고 편집으로 표정을 바꾼다.
표정마다 새로 생성하면 매번 얼굴이 달라져 같은 인물로 안 보인다.


In [ ]:
# 표정 세트는 만들지 않는다.
# HiDreamImagePipeline 은 레퍼런스 이미지를 못 받으므로 "같은 인물의 표정만 변경"
# 이 원리적으로 불가능하다. 예전 코드는 인물 묘사가 빠진 프롬프트로 생성해서
# 매번 다른 사람이 나오는데도 에러가 없었다. 인물당 1장만 만든다.
for scen, v in P.items():
    for cid, prompt in v["portraits"].items():
        gen(prompt, seed=0).save(out_path(scen, "portraits", f"{cid}_p0.png"))
        print("  ", scen, cid)

In [ ]:
del pipe
free()


## 8. 합성 확인

`home_place_id` 로 짝을 맞춰 배경 위에 인물을 얹는다.
모델 선택 기준이 "붙여놨을 때 한 그림인가"이므로 이걸 눈으로 봐야 한다.


In [ ]:
from IPython.display import display
from PIL import Image

def composite(scen, w=900):
    d = DATA[scen]
    for c in d["cast"]:
        bg = OUT_DIR / scen / "places" / f"{c['home_place_id']}.png"
        ch = OUT_DIR / scen / "portraits" / f"{c['id']}_p0.png"
        if not (bg.exists() and ch.exists()):
            continue
        b = Image.open(bg).convert("RGB").resize((w, int(w * 0.6)))
        s = int(b.height * 0.85)
        b.paste(Image.open(ch).convert("RGB").resize((s, s)), (int(w * 0.06), b.height - s))
        out = out_path(scen, "composites", f"{c['id']}.png")
        b.save(out)
        print(f"{c['id']} ({c['occupation']}) @ {c['home_place_id']}")
        display(b)

# 시나리오 하나만 골라 눈으로 확인한다 (전부 다 볼 필요는 없다).
demo_scen = next(iter(DATA))
composite(demo_scen)


## 9. 확인 항목

1. 장소끼리 같은 세계로 보이는가 (STYLE 문장이 앵커 역할을 했는가)
2. **시대·문화가 맞는가** — 흥부놀부가 조선인가, 중국/일본풍으로 뭉개졌는가
3. **심문 UI에 얹었을 때 대사가 읽히는가** — 배경이 화려할수록 불리하다
4. 원작 IP 디자인이 새어나오지 않았는가
5. FLUX 쪽과 비교할 때, FLUX 는 world_ref 를 레퍼런스 이미지로 걸 수 있고
   여기는 문장으로만 맞춘다. 일관성 차이는 이 조건 차이를 감안하고 볼 것.